In [1]:
from pathlib import Path

# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

In [2]:
from collections import defaultdict
import zipfile
import gc
import pandas as pd

def build_dataset(folder_name):

    # Each key is a table name (event, venue, tournament, ...)
    # Each value is a list of daily DataFrames
    tables = defaultdict(list)

    # Find all ZIP files and sort them by date
    zip_files = sorted(zip_folder.glob("*.zip"))

    # Iterate over ZIP files
    for i, zip_path in enumerate(zip_files, start=1):

        # Log the file we are working on
        print(f"{i}/{len(zip_files)}")

        # Store daily data separately for each table
        daily_tables = defaultdict(list)

        with zipfile.ZipFile(zip_path) as z:
            for file in z.namelist():
                if file.endswith(".parquet") and f"/{folder_name}/" in file:

                    # Extract table name
                    filename = file.split("/")[-1].replace(".parquet", "")
                    table_name = filename.rsplit("_", 1)[0]

                    # Read parquet
                    with z.open(file) as f:
                        df = pd.read_parquet(f)

                    # Add date column
                    df["date"] = zip_path.stem

                    # Store in today's table
                    daily_tables[table_name].append(df)

        # Concat each table for this day
        for table_name, dfs in daily_tables.items():

            day_df = pd.concat(dfs, ignore_index=True)
            tables[table_name].append(day_df)

        # Free RAM
        del daily_tables
        gc.collect()

    # Concat all files
    for table_name, chunks in tables.items():
        final_df = pd.concat(chunks, ignore_index=True)
        final_df.to_parquet( zip_folder / f"{table_name}_all.parquet", index=False)

        # Get size to ensure accuracy
        print(final_df.shape)

        # Free RAM
        del final_df
        gc.collect()

    # Free RAM
    del tables
    gc.collect()

    print("Finished!")

In [ ]:
# Match tables (10 tables)
build_dataset("raw_match_parquet")

In [ ]:
# Odds
build_dataset("raw_odds_parquet")

In [ ]:
# Statistics
build_dataset("raw_statistics_parquet")

In [ ]:
# Votes
build_dataset("raw_votes_parquet")

In [ ]:
# Tennis Power
build_dataset("raw_tennis_power_parquet")

In [ ]:
# Point by Point
build_dataset("raw_point_by_point_parquet")